# Recommendation Systems - Complete Usage Examples

This notebook demonstrates how to use Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [1]:
import pandas as pd
import numpy as np
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [2]:
loader = MovieLensDataLoader()
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=500)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading data from cache: data/processed/movie_metadata.csv


Movies: 8921
Ratings: 100836
Users: 610


## 2. Content-Based Filtering

In [3]:
config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    keywords_weight=0.6,
    numerical_weight=0.1,
    similarity_threshold=0.15,
    top_k_default=10
)

cb_model = ContentBasedRecommender(config=config)
cb_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Content-Based model trained successfully!")

Content-Based model trained successfully!


### 2.1 Get Recommendations for a User

In [19]:
user_id = 444
user_history = ratings_df[ratings_df['userId'] == user_id]
watched_items = set(user_history['movieId'].values)
liked_items = set(user_history[user_history['rating'] >= 4.0]['movieId'].values)

print(f"User {user_id} has watched {len(watched_items)} movies")
print(f"User {user_id} liked {len(liked_items)} movies (rating >= 4.0)")

User 444 has watched 42 movies
User 444 liked 27 movies (rating >= 4.0)


In [20]:
recommendations = cb_model.get_top_k_with_titles(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Recommendations for User {user_id}:")
print("=" * 80)
for i, rec in enumerate(recommendations, 1):
    print(f"{i:2}. {rec['title']:<50} (Score: {rec['score']:.4f})")


Top 10 Recommendations for User 444:
 1. GoodFellas                                         (Score: 0.3255)
 2. Shutter Island                                     (Score: 0.2926)
 3. The Godfather Part II                              (Score: 0.2842)
 4. Looper                                             (Score: 0.2835)
 5. Raging Bull                                        (Score: 0.2789)
 6. Insomnia                                           (Score: 0.2788)
 7. The Godfather Part III                             (Score: 0.2787)
 8. Scarface                                           (Score: 0.2734)
 9. Léon: The Professional                             (Score: 0.2690)
10. The Wolf of Wall Street                            (Score: 0.2639)


### 2.2 Explain Recommendations

In [21]:
movie_id_to_title = dict(zip(movies_df['movieId'], movies_df['title']))

if recommendations:
    rec_movie_id = recommendations[0]['movieId']
    rec_title = recommendations[0]['title']
    
    print(f"Why was '{rec_title}' recommended?")
    print("=" * 80)
    
    reasons = cb_model.explain_recommendation(
        movie_id=rec_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

Why was 'GoodFellas' recommended?
1. Casino (Similarity: 0.4051)
2. Taxi Driver (Similarity: 0.3610)
3. Heat (Similarity: 0.1908)
4. Carlito's Way (Similarity: 0.1629)


### 2.3 Find Similar Movies

In [22]:
target_movie_id = 4.0
target_movie_title = movies_df[movies_df['movieId'] == target_movie_id]['title'].values[0]

print(f"Movies similar to '{target_movie_title}':")
print("=" * 80)

similar_movies = cb_model.get_similar_movies(
    movie_id=target_movie_id,
    top_n=10
)

for i, movie in enumerate(similar_movies, 1):
    title = movie_id_to_title.get(movie['movie_id'], f"ID: {movie['movie_id']}")
    print(f"{i:2}. {title:<50} (Similarity: {movie['similarity']:.4f})")

Movies similar to 'Waiting to Exhale':
 1. How Stella Got Her Groove Back                     (Similarity: 0.2120)
 2. Nothing in Common                                  (Similarity: 0.2004)
 3. What's Love Got to Do with It                      (Similarity: 0.1786)
 4. 12 Chairs                                          (Similarity: 0.1687)
 5. Winter's Tale                                      (Similarity: 0.1668)
 6. Cinderella                                         (Similarity: 0.1658)
 7. Pride and Prejudice                                (Similarity: 0.1642)
 8. The First Wives Club                               (Similarity: 0.1609)
 9. I Love You, Beth Cooper                            (Similarity: 0.1546)
10. The Four Seasons                                   (Similarity: 0.1530)


### 2.4 Predict Scores for Specific Movies

In [23]:
candidate_movies = [1.0, 2.0, 3.0, 4.0, 5.0]
scores = cb_model.predict_scores(user_id=user_id, item_ids=candidate_movies)

print(f"Predicted scores for User {user_id}:")
print("=" * 80)
for movie_id, score in zip(candidate_movies, scores):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    print(f"{title:<50} -> Score: {score:.4f}")

Predicted scores for User 444:
Toy Story                                          -> Score: 0.1767
Jumanji                                            -> Score: 0.1445
ID: 3.0                                            -> Score: 0.0000
Waiting to Exhale                                  -> Score: 0.0892
ID: 5.0                                            -> Score: 0.0000


## 3. Collaborative Filtering

In [24]:
cf_model = CollaborativeFiltering(
    k_components=50,
    random_state=42
)
cf_model.fit(df_ratings=ratings_df)
print("Collaborative Filtering model trained successfully!")

Collaborative Filtering model trained successfully!


### 3.1 Get Recommendations

In [25]:
cf_recommendations = cf_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 CF Recommendations for User {user_id}:")
print("=" * 80)
for i, movie_id in enumerate(cf_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = cf_model.predict_score(user_id, movie_id)
    print(f"{i:2}. {title:<50} (Predicted Rating: {score:.4f})")


Top 10 CF Recommendations for User 444:
 1. Ran                                                (Predicted Rating: 4.3315)
 2. Grave of the Fireflies                             (Predicted Rating: 4.3544)
 3. Touch of Evil                                      (Predicted Rating: 4.3867)
 4. Kelly's Heroes                                     (Predicted Rating: 4.2747)
 5. Rebel Without a Cause                              (Predicted Rating: 4.2124)
 6. ID: 1719                                           (Predicted Rating: 4.2545)
 7. A Streetcar Named Desire                           (Predicted Rating: 4.4307)
 8. Hamlet                                             (Predicted Rating: 4.2316)
 9. The Outlaw Josey Wales                             (Predicted Rating: 4.3245)
10. Three Colors: Red                                  (Predicted Rating: 4.2208)


### 3.2 Explain CF Recommendations

In [26]:
if cf_recommendations:
    cf_movie_id = cf_recommendations[0]
    cf_title = movie_id_to_title.get(cf_movie_id, f"ID: {cf_movie_id}")
    
    print(f"Why was '{cf_title}' recommended (CF)?")
    print("=" * 80)
    
    cf_reasons = cf_model.explain_recommendation(
        movie_id=cf_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(cf_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

Why was 'Ran' recommended (CF)?
1. ID: 356 (Similarity: 0.3134)
2. ID: 273 (Similarity: 0.2780)
3. Casino (Similarity: 0.2132)
4. La Haine (Similarity: 0.1660)
5. The Silence of the Lambs (Similarity: 0.1421)


## 4. Hybrid Model

In [ ]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.8
)

hybrid_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Hybrid model trained successfully!")

Hybrid model trained successfully!


### 4.1 Get Hybrid Recommendations

In [28]:
hybrid_recommendations = hybrid_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Hybrid Recommendations for User {user_id} (alpha=0.5):")
print("=" * 80)
for i, movie_id in enumerate(hybrid_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = hybrid_model.predict_scores(user_id, [movie_id])[0]
    print(f"{i:2}. {title:<50} (Score: {score:.4f})")


Top 10 Hybrid Recommendations for User 444 (alpha=0.5):
 1. GoodFellas                                         (Score: 0.0000)
 2. The Godfather Part II                              (Score: 0.0000)
 3. Raging Bull                                        (Score: 0.0000)
 4. The Dark Knight                                    (Score: 0.0000)
 5. Shutter Island                                     (Score: 0.0000)
 6. The Departed                                       (Score: 0.0000)
 7. Léon: The Professional                             (Score: 0.0000)
 8. Memento                                            (Score: 0.0000)
 9. Reservoir Dogs                                     (Score: 0.0000)
10. The Lord of the Rings: The Fellowship of the Ring  (Score: 0.0000)


### 4.2 Explain Hybrid Recommendations

In [29]:
if hybrid_recommendations:
    hybrid_movie_id = hybrid_recommendations[0]
    hybrid_title = movie_id_to_title.get(hybrid_movie_id, f"ID: {hybrid_movie_id}")
    
    print(f"Why was '{hybrid_title}' recommended (Hybrid)?")
    print("=" * 80)
    
    hybrid_reasons = hybrid_model.explain_recommendation(
        movie_id=hybrid_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(hybrid_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Combined Similarity: {reason['similarity']:.4f})")

Why was 'GoodFellas' recommended (Hybrid)?
1. The Shawshank Redemption (Combined Similarity: 0.3994)
2. Taxi Driver (Combined Similarity: 0.3610)
3. Ed Wood (Combined Similarity: 0.3207)
4. Casino (Combined Similarity: 0.2915)
5. Four Weddings and a Funeral (Combined Similarity: 0.2506)


## 5. Compare All Three Models

In [30]:
print(f"\nComparison of Top 5 Recommendations for User {user_id}:")
print("=" * 120)
print(f"{'Rank':<6}{'Content-Based':<35}{'Collaborative':<35}{'Hybrid':<35}")
print("-" * 120)

for i in range(5):
    cb_title = movie_id_to_title.get(recommendations[i]['movieId'], "N/A")[:33] if i < len(recommendations) else "N/A"
    cf_title = movie_id_to_title.get(cf_recommendations[i], "N/A")[:33] if i < len(cf_recommendations) else "N/A"
    hybrid_title = movie_id_to_title.get(hybrid_recommendations[i], "N/A")[:33] if i < len(hybrid_recommendations) else "N/A"
    
    print(f"{i+1:<6}{cb_title:<35}{cf_title:<35}{hybrid_title:<35}")


Comparison of Top 5 Recommendations for User 444:
Rank  Content-Based                      Collaborative                      Hybrid                             
------------------------------------------------------------------------------------------------------------------------
1     GoodFellas                         Ran                                GoodFellas                         
2     Shutter Island                     Grave of the Fireflies             The Godfather Part II              
3     The Godfather Part II              Touch of Evil                      Raging Bull                        
4     Looper                             Kelly's Heroes                     The Dark Knight                    
5     Raging Bull                        Rebel Without a Cause              Shutter Island                     


## 6. Save and Load Models

In [31]:
cb_model.save_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model saved!")

Content-Based model saved!


In [32]:
cb_model_loaded = ContentBasedRecommender(config=config)
cb_model_loaded.load_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model loaded!")

test_recommendations = cb_model_loaded.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=5
)
print(f"Test: Generated {len(test_recommendations)} recommendations with loaded model")

Content-Based model loaded!
Test: Generated 5 recommendations with loaded model


## 7. Full User Profile Display

In [33]:
cb_model.show_user_profile_and_recommendations(
    user_id=user_id,
    ratings_df=ratings_df,
    movies_df=movies_df,
    k=10,
    top_rated_count=5,
    reasons_count=3
)

USER PROFILE: 444

General Statistics:
   - Total ratings: 42
   - Average rating: 3.81 / 5.0
   - Unique movies rated: 42

TOP 5 FAVORITE MOVIES (Rating >= 4.0):
--------------------------------------------------------------------------------
 1. La Haine                                           [Rating: 5.0/5.0]
 2. The Postman                                        [Rating: 5.0/5.0]
 3. The Usual Suspects                                 [Rating: 5.0/5.0]
 4. Ed Wood                                            [Rating: 5.0/5.0]
 5. Taxi Driver                                        [Rating: 5.0/5.0]

PERSONALIZED RECOMMENDATIONS FOR USER 444

 1. GoodFellas
    Predicted Relevance Score: 0.3255
    Recommended because you liked:
       - Casino (You rated: 4.0/5.0) [Similarity: 0.4051]
       - Taxi Driver (You rated: 5.0/5.0) [Similarity: 0.3610]
       - Heat (You rated: 4.0/5.0) [Similarity: 0.1908]

 2. Shutter Island
    Predicted Relevance Score: 0.2926
    Recommended because 